*Figure notebook of our analysis (IDIBAPS cFos brains). It reads the project settings from the environment variable `CFOS_MAPPER_SETTINGS` (or `./settings.json`); set it before running:* `%env CFOS_MAPPER_SETTINGS=/path/to/settings.json`

# Step 6 - cell density per brain region, two groups of mice

Kernel: **cellfinder-latest (1.11)**

Uses the cells per atlas region of every brain (`cfos-mapper count`, which writes
`results/<brain>/full_brain/<run_name>/regions/cells_per_region.csv`) and compares two groups of mice
(here: the same instrumental task guided by light, *Visual*, or by sound, *Auditory*).

Regions: the ~300 **Beryl regions** of the International Brain Laboratory (IBL): one level of the Allen atlas,
fine enough to see e.g. the layers of the superior colliculus apart, coarse enough to have enough cells per region.
No Beryl region contains another one, so every cell is counted once.

It makes:

0. cell density of the whole brain, per mouse (to check that the mice are comparable)
1. the regions that differ most between the groups: density of every mouse, as box plots (like a bar chart of
   cell counts per region, but with the spread of the mice and in cells per mm³)
2. a volcano plot: difference between the groups against its p-value, for all regions
3. the same box plots for all regions, one figure per major division of the brain (isocortex, thalamus, ...)
4. atlas maps: sections of the atlas (sagittal by default; horizontal or coronal with `plane`) with every region
   coloured by its density, average of each group; the two groups side by side; and the difference between the groups
5. reference figures of the Beryl regions: the same atlas sections with the acronym of every region, and the list of
   all regions (acronym and full name) by major division

Tables (CSV) and figures (PDF, editable in Illustrator, and PNG for slides) are saved in
`figures/regions_<run_name>/`.

## Settings

- `run_name`: the network used for the whole brains (= folder name in `results/<brain>/full_brain/`).
- `groups`: the mice of each group, **as the brains are named in the results folder** (these names are also used
  in the figures). The first group is the reference: a ratio > 1 (log2 ratio > 0) means more cells in the second
  group.
- `normalise`: if `True`, every density is divided by the density of the whole brain of that mouse. This removes
  differences that touch the whole brain (staining, clearing, imaging, detection) and keeps only *where* the cells
  are. Look at figure 0 first: if one mouse has many more cells everywhere, normalising helps.
- `min_volume_mm3`: regions smaller than this (average over the mice) are left out; a few voxels of the 25 µm
  atlas do not give a reliable density.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.cm import ScalarMappable
from matplotlib.collections import PolyCollection
from matplotlib.colors import Normalize
from scipy import stats
from skimage import measure
import cfos_mapper as cf
from cfos_mapper.figures import regions as fr
from cfos_mapper.figures import style as fs

fs.apply_style()
settings = cf.load_settings()
results = Path(settings['results_folder'])

# the network used in 4_run_brain.py (= folder name in results/<brain>/full_brain/)
run_name = 'network_20261023'

# brains as they are named in the results folder; the first group is the reference (ratio = second / first)
groups = {
    'Visual': ['ACV011', 'ACV012', 'ACV013', 'ACV014', 'ACV015', 'ACV016'],
    'Auditory': ['ACV017', 'ACV018', 'ACV019', 'ACV020', 'ACV021', 'ACV022'],
}
group_colors = {'Visual': fs.BLUE, 'Auditory': fs.ORANGE}
first, second = groups

normalise = False      # True: densities divided by the density of the whole brain of the mouse
min_volume_mm3 = 0.01  # regions smaller than this (average over the mice) are left out

out = results.parent / 'figures' / f'regions_{run_name}'

## Load the cells per region of every mouse

`cells_per_region.csv` has one row per atlas region (every level, from single layers up to the whole brain,
`root`), with the cells in it, its volume in this brain (from the registered atlas) and the density. Brains that are
not counted yet are left out with a message, so the notebook also runs with part of the mice.

The table below has one row per mouse: all its cells in the atlas, the volume of the brain and the density of the
whole brain.

In [ ]:
tables = []
for group, brains in groups.items():
    for brain in brains:
        file = results / brain / 'full_brain' / run_name / 'regions' / 'cells_per_region.csv'
        if file.exists():
            tables.append(pd.read_csv(file).assign(group=group, mouse=brain))
        else:
            print(f'{brain}: left out, not counted yet (no {file})')
all_regions = pd.concat(tables, ignore_index=True)
mice = {group: [b for b in brains if b in set(all_regions.mouse)] for group, brains in groups.items()}

whole_brain = all_regions[all_regions.acronym == 'root'].set_index('mouse').loc[mice[first] + mice[second]]
overview = pd.DataFrame({
    'group': whole_brain.group,
    'cells': whole_brain.cells,
    'brain volume (mm³)': whole_brain.volume_mm3.round(1),
    'cells per mm³': whole_brain.cells_per_mm3.round(0),
})
overview

## Density in the Beryl regions

`fr.beryl_regions(atlas)` lists the Beryl regions in Allen order (isocortex from front to back, olfactory areas,
hippocampus, ..., cerebellum), with the major division each one belongs to. The Beryl regions are taken by their
Allen structure id (the list is in `cfos_mapper/figures/regions.py`).

Three tables with one row per region and one column per mouse: `cells`, `volume` (mm³) and
`density` = cells / volume. `density` is empty (NaN) where a region is not in a brain at all. Then regions smaller
than `min_volume_mm3` are left out. The last column of the overview says how many of the cells of each mouse are in
a Beryl region (the others are in fiber tracts, ventricles, or in voxels the atlas labels only with a big region).

Saved: `density_per_region.csv` (one row per region, one column per mouse).

In [ ]:
atlas = cf.load_atlas(settings)
beryl = fr.beryl_regions(atlas).set_index('structure_id')
in_beryl = all_regions[all_regions.structure_id.isin(beryl.index)]

def per_mouse(column):
    """One row per Beryl region (Allen order), one column per mouse (first group, then second group)."""
    table = in_beryl.pivot(index='structure_id', columns='mouse', values=column)
    return table.reindex(index=beryl.index, columns=mice[first] + mice[second])

cells = per_mouse('cells').fillna(0)
volume = per_mouse('volume_mm3')
density = cells / volume.where(volume > 0)
if normalise:
    density = density / whole_brain.cells_per_mm3
density_unit = 'density / whole-brain density' if normalise else 'cells per mm³'

big_enough = volume.mean(axis=1) >= min_volume_mm3
print(f'{len(beryl)} Beryl regions; {(~big_enough).sum()} smaller than {min_volume_mm3} mm³ left out: '
      + ', '.join(beryl.acronym[~big_enough]))
density = density[big_enough]

out.mkdir(parents=True, exist_ok=True)
beryl.join(density, how='inner').to_csv(out / 'density_per_region.csv')
overview['cells in Beryl regions (%)'] = (100 * cells.sum() / whole_brain.cells).round(1)
overview

## Statistics per region

For every region, the two groups are compared:

- `mean <group>`: average density of the mice of each group.
- `change (%)` and `log2_ratio`: how much higher the second group is than the first, in percent and as log2 of the
  ratio of the means (+1 = twice as many cells per mm³ in the second group, -1 = half).
- `p`: Welch's t-test on log10(density). Cell densities are skewed (a few mice with many cells), the log makes
  them close to normal, so the t-test is fair; Welch's version does not assume that both groups vary the same.
- `q`: `p` corrected for testing ~300 regions at once (Benjamini-Hochberg false discovery rate). With 6 mice per
  group, expect few regions with q < 0.05; use `p` to rank and `q` to tell what survives the correction.
- `p_mann_whitney`: the same comparison without assuming anything about the distribution (ranks only). With 6 + 6
  mice its smallest possible value is 0.002, so it is coarse; it is there to check that `p` is not driven by one mouse.

Zeros (a region without cells in a mouse) are replaced by half the smallest density of the table before taking
logs and ratios.

Saved: `statistics_per_region.csv` (sorted by `p`). The cell shows the 20 regions with the smallest `p`.

In [ ]:
values = density.to_numpy()
floor = values[values > 0].min() / 2           # replaces zeros before taking logs and ratios
log_density = np.log10(density.clip(lower=floor))
reference, other = log_density[mice[first]].to_numpy(), log_density[mice[second]].to_numpy()

statistics = beryl.loc[density.index].copy()
statistics[f'mean {first}'] = density[mice[first]].mean(axis=1)
statistics[f'mean {second}'] = density[mice[second]].mean(axis=1)
ratio = statistics[f'mean {second}'].clip(lower=floor) / statistics[f'mean {first}'].clip(lower=floor)
statistics['change (%)'] = 100 * (ratio - 1)
statistics['log2_ratio'] = np.log2(ratio)
statistics['p'] = stats.ttest_ind(other, reference, axis=1, equal_var=False, nan_policy='omit').pvalue
tested = statistics.p.notna()
statistics.loc[tested, 'q'] = stats.false_discovery_control(statistics.p[tested])
statistics['p_mann_whitney'] = stats.mannwhitneyu(other, reference, axis=1, nan_policy='omit').pvalue

statistics.sort_values('p').to_csv(out / 'statistics_per_region.csv')
print(f'{tested.sum()} regions tested: {(statistics.p < 0.05).sum()} with p < 0.05, '
      f'{(statistics.q < 0.05).sum()} with q < 0.05')
statistics.sort_values('p').head(20).round(
    {f'mean {first}': 0, f'mean {second}': 0, 'change (%)': 0, 'log2_ratio': 2, 'p': 4, 'q': 3, 'p_mann_whitney': 4})

## Plotting helpers

- `box`: one group of mice in one region: dots = the mice, box = the middle half of them (25-75%), thick line = the
  median, whiskers = the lowest and the highest mouse. With 6 mice a box plot hides little, and every mouse stays
  visible.
- `region_box_plots`: one row per region (full name and acronym), the two groups as two boxes, density on a log
  axis (densities differ by more than 100x between regions). All figures use the same axis range, so they can be
  compared. Zeros are drawn at `floor` (they do not exist on a log axis).
- `p_values`: p and q of every region, written right of the plot.
- `group_legend`: the colour of each group, with the number of mice.

In [ ]:
shown = density.clip(lower=floor)                  # zeros cannot be drawn on a log axis
density_range = (shown.min().min() / 1.5, shown.max().max() * 1.5)

def box(ax, position, values, color, horizontal=True, width=0.32, jitter=0.07):
    """Dots = mice, box = middle half (25-75%), thick line = median, whiskers = lowest and highest mouse."""
    values = np.asarray(values, float)
    values = values[~np.isnan(values)]
    if len(values) == 0:
        return
    q1, median, q3 = np.percentile(values, [25, 50, 75])
    half = width / 2

    def draw(value_axis, across, **style):
        x, y = (value_axis, across) if horizontal else (across, value_axis)
        ax.plot(x, y, color=color, solid_capstyle='butt', **style)

    corners = np.array([-half, -half, half, half, -half]) + position
    x, y = ([q1, q3, q3, q1, q1], corners) if horizontal else (corners, [q1, q3, q3, q1, q1])
    ax.fill(x, y, color=color, alpha=0.18, linewidth=0)
    draw([q1, q3, q3, q1, q1], corners, linewidth=1)
    draw([median, median], [position - half, position + half], linewidth=2.2)
    draw([values.min(), q1], [position, position], linewidth=1)
    draw([q3, values.max()], [position, position], linewidth=1)
    spread = position + np.random.default_rng(0).uniform(-jitter, jitter, len(values))
    x, y = (values, spread) if horizontal else (spread, values)
    ax.scatter(x, y, s=14, color=color, edgecolors=fs.SURFACE, linewidths=0.6, zorder=3)

def region_box_plots(ax, table):
    """Density of every mouse in the regions of `table` (index = structure_id), first row at the top."""
    for row, structure_id in enumerate(table.index):
        for offset, group in [(-0.19, first), (0.19, second)]:
            box(ax, row + offset, shown.loc[structure_id, mice[group]], group_colors[group])
    ax.set_yticks(range(len(table)), [f'{name} ({acronym})' for name, acronym in zip(table.name, table.acronym)])
    ax.set_ylim(len(table) - 0.5, -0.5)
    ax.set_xscale('log')
    ax.set_xlim(*density_range)
    ax.set_xlabel(density_unit)
    ax.grid(True, axis='x')
    ax.grid(False, axis='y')
    ax.tick_params(axis='y', length=0, labelsize=9)

def p_values(ax, table):
    """p (t-test) and q (false discovery rate) of every region, right of the plot."""
    for row, (p, q) in enumerate(zip(table.p, table.q)):
        ax.text(1.01, row, f'p {p:.2g}   q {q:.2g}', transform=ax.get_yaxis_transform(), va='center',
                fontsize=8, color=fs.INK_2 if p < 0.05 else fs.MUTED)

def group_legend(ax):
    handles = [plt.Line2D([], [], marker='o', linestyle='', markersize=7, color=group_colors[group],
                          label=f'{group} (n = {len(mice[group])})') for group in groups]
    ax.legend(handles=handles, loc='lower left', bbox_to_anchor=(0, 1), ncols=2, borderaxespad=0.3)

## 0. Cell density of the whole brain, per mouse

One bar per mouse, in the colour of its group. If the groups differ here, part of the differences per region come
from the whole brain (or from staining / imaging); if one mouse is far from the others of its group, consider
`normalise = True` or check that brain.

In [ ]:
fig, ax = plt.subplots(figsize=(1.5 + 0.55 * len(whole_brain), 4.2))
bars = ax.bar(whole_brain.index, whole_brain.cells_per_mm3, width=0.65,
              color=[group_colors[g] for g in whole_brain.group])
fs.bar_labels(ax, bars, '{:,.0f}')
ax.set_ylabel('cells per mm³')
ax.tick_params(axis='x', length=0)
plt.setp(ax.get_xticklabels(), rotation=45, ha='right', rotation_mode='anchor')
ax.margins(y=0.12)
group_legend(ax)
fs.title(fig, 'Cell density of the whole brain')
fs.save(fig, out, '0_whole_brain_density')

## 1. The regions that differ most between the groups

The `n_top` regions with the smallest p-value (`rank_by = 'p'`), or with the biggest difference between the group
means in either direction (`rank_by = 'ratio'`; this one also shows big differences that are not consistent across
mice). One row per region, the best first. p and q on the right; in dark ink when p < 0.05.

In [ ]:
n_top = 30
rank_by = 'p'    # 'p': smallest p-value first; 'ratio': biggest difference (either direction) first

if rank_by == 'p':
    top = statistics.sort_values('p').head(n_top)
else:
    top = statistics.loc[statistics.log2_ratio.abs().sort_values(ascending=False).index].head(n_top)

fig, ax = plt.subplots(figsize=(9, 0.3 * len(top) + 1.6))
region_box_plots(ax, top)
p_values(ax, top)
group_legend(ax)
fs.title(fig, f'The {len(top)} regions that differ most between {first} and {second} (ranked by {rank_by})')
fs.save(fig, out, f'1_top_regions_by_{rank_by}')

## 2. Volcano plot: all regions at once

Every dot is a region. Left-right: how much the groups differ (log2 of the ratio of the means; left = more cells in
the first group, right = more in the second). Up: how consistent the difference is across the mice (-log10 p; the
dashed line is p = 0.05, the dotted one q = 0.05 if any region reaches it). Coloured: p < 0.05, in the colour of the
group with more cells. The `n_labels` regions with the smallest p are named.

In [ ]:
n_labels = 15

x, y = statistics.log2_ratio, -np.log10(statistics.p)
significant = statistics.p < 0.05
fig, ax = plt.subplots(figsize=(7.5, 6))
ax.scatter(x[~significant], y[~significant], s=16, color=fs.GRAY, edgecolors=fs.SURFACE, linewidths=0.5,
           label='p ≥ 0.05')
for group, side in [(first, x < 0), (second, x > 0)]:
    ax.scatter(x[significant & side], y[significant & side], s=22, color=group_colors[group],
               edgecolors=fs.SURFACE, linewidths=0.5, label=f'p < 0.05, more cells in {group}')
ax.axhline(-np.log10(0.05), color=fs.MUTED, linewidth=0.8, linestyle='--')
if (statistics.q < 0.05).any():
    ax.axhline(-np.log10(statistics.p[statistics.q < 0.05].max()), color=fs.MUTED, linewidth=0.8, linestyle=':')
ax.axvline(0, color=fs.AXIS, linewidth=0.8)
for _, region in statistics.nsmallest(n_labels, 'p').iterrows():
    ax.annotate(region.acronym, (region.log2_ratio, -np.log10(region.p)), xytext=(4, 3),
                textcoords='offset points', fontsize=8, color=fs.INK_2)
limit = np.nanmax(np.abs(x)) * 1.1
ax.set_xlim(-limit, limit)
ax.set_ylim(0, None)
ax.set_xlabel(f'log2 ratio of the mean density, {second} / {first}')
ax.set_ylabel('-log10 p')
ax.grid(True, axis='both')
ax.legend(loc='upper left', fontsize=9)
fs.title(fig, f'{first} vs {second}: {len(statistics)} regions')
fs.save(fig, out, '2_volcano')

## 3. All regions, one figure per major division

The same box plots as in figure 1, for every region, in Allen order, one figure per major division of the brain
(the IBL "Cosmos" level: isocortex, olfactory areas, hippocampal formation, cortical subplate, cerebral nuclei,
thalamus, hypothalamus, midbrain, hindbrain, cerebellum). All use the same density axis.

In [ ]:
for number, division in enumerate(fr.COSMOS):
    table = statistics[statistics.division == division]
    if table.empty:
        continue
    fig, ax = plt.subplots(figsize=(9, 0.3 * len(table) + 1.6))
    region_box_plots(ax, table)
    p_values(ax, table)
    group_legend(ax)
    fs.title(fig, f'{fr.COSMOS_NAMES[division]}: {len(table)} regions')
    fs.save(fig, out, f'3_{number}_{division}')

## 4. Atlas maps

Sections of the Allen atlas (25 µm), every Beryl region filled with a value; white = tissue without a value
(fiber tracts, ventricles, regions left out). The regions are drawn as outlines (vector shapes), so the PDF stays
sharp at any zoom.

- `plane`: the cut. `'sagittal'` (longitudinal, side view: front on the left, top up; the default), `'horizontal'`
  (longitudinal, top view: front up) or `'coronal'` (front view, top up). The figures have the plane in their file
  name, so you can make all three: change `plane` and run the cells from here on again.
- `n_sections` sections, spread evenly: sagittal from the side of the brain to the midline (one hemisphere; the
  values are the average of both hemispheres anyway), horizontal from top to bottom, coronal from front to back.
  To choose them yourself, set `sections` to atlas planes (steps of 25 µm) at the end of this cell.
- `section_shapes(index)` turns the regions of one section into polygons (biggest first, so that a region inside
  another one is drawn on top of it). With `split=True` (horizontal and coronal) every region is cut at the
  midline, to colour the two halves differently.
- `draw_section` fills the polygons with the colour of their value, in gray outlines. `bold` regions get a dark
  outline.

In [ ]:
plane = 'sagittal'   # 'sagittal' (side view), 'horizontal' (top view) or 'coronal' (front view)
n_sections = 8

annotation = np.asarray(atlas.annotation)          # axes: front -> back, top -> bottom, right -> left
assert atlas.metadata['orientation'] == 'asr', 'the sections below expect an atlas in brainglobe orientation asr'
atlas_mm = atlas.resolution[0] / 1000
middle = annotation.shape[2] / 2                   # the midline, between right and left

def cut(volume, index):
    """
    One section of an atlas volume as an image. sagittal: rows top -> bottom, columns front -> back;
    horizontal: rows front -> back, columns right -> left; coronal: rows top -> bottom, columns right -> left.
    """
    if plane == 'sagittal':
        return volume[:, :, index].T
    if plane == 'horizontal':
        return volume[:, index, :]
    return volume[index]

axis = {'coronal': 0, 'horizontal': 1, 'sagittal': 2}[plane]
present = np.flatnonzero(annotation.any(axis=tuple(a for a in range(3) if a != axis)))
if plane == 'sagittal':   # one hemisphere, from the side to just next to the midline
    sections = np.linspace(present[0], middle - 2, n_sections + 1)[1:].round().astype(int)
else:
    sections = np.linspace(present[0], present[-1], n_sections + 2)[1:-1].round().astype(int)

def position(index):
    """Where a section is, in mm (from the midline, the top or the front of the atlas)."""
    return abs(middle - index) * atlas_mm if plane == 'sagittal' else index * atlas_mm

position_label = {'sagittal': 'mm from the midline', 'horizontal': 'mm from the top of the atlas',
                  'coronal': 'mm from the front of the atlas'}[plane]

# crop all sections to the same box around the brain
tissue = np.any([cut(annotation, index) > 0 for index in sections], axis=0)
rows, columns = np.flatnonzero(tissue.any(axis=1)), np.flatnonzero(tissue.any(axis=0))
crop = (slice(rows[0], rows[-1] + 1), slice(columns[0], columns[-1] + 1))
height, width = crop[0].stop - crop[0].start, crop[1].stop - crop[1].start
panel = (2.2, 2.2 * height / width)                # size of one section in the figures (inches)

def section_shapes(index, split=False):
    """
    The regions of one section as polygons: list of (structure_id, half, (n, 2) x-y points), biggest first.
    structure_id 0 = tissue in no Beryl region; half = 1 / 2 (the two halves of the image) when split, else 0.
    """
    section = cut(annotation, index)[crop]
    labels = fr.to_beryl(section, atlas)
    inside = section > 0
    column = np.arange(crop[1].start, crop[1].stop)[None, :].repeat(labels.shape[0], axis=0)
    halves = np.where(column < middle, 1, 2) if split else np.zeros(labels.shape, int)
    shapes = []
    for structure_id in np.unique(labels[inside]):
        for half in np.unique(halves[inside & (labels == structure_id)]):
            mask = np.pad(inside & (labels == structure_id) & (halves == half), 1)
            for contour in measure.find_contours(mask.astype(float), 0.5):
                y, x = contour[:, 0] - 1, contour[:, 1] - 1
                area = 0.5 * np.sum(x * np.roll(y, -1) - np.roll(x, -1) * y)
                if area > 0:                       # outlines (holes go the other way round)
                    shapes.append((area, structure_id, half, np.c_[x, y]))
    shapes.sort(key=lambda shape: -shape[0])
    return [shape[1:] for shape in shapes]

shapes = {index: section_shapes(index) for index in sections}
split_shapes = {} if plane == 'sagittal' else {index: section_shapes(index, split=True) for index in sections}

def draw_section(ax, polygons, color_of, bold=()):
    """Polygons filled with color_of(structure_id, half) (None = no value: white), gray outlines."""
    colors = [color_of(structure_id, half) if structure_id else None for structure_id, half, _ in polygons]
    ax.add_collection(PolyCollection([points for _, _, points in polygons],
                                     facecolors=[c if c is not None else fs.SURFACE for c in colors],
                                     edgecolors=fs.MUTED, linewidths=0.35))
    outlined = [points for structure_id, _, points in polygons if structure_id in bold]
    if outlined:
        ax.add_collection(PolyCollection(outlined, facecolors='none', edgecolors=fs.INK, linewidths=1.0))
    ax.set_xlim(-1, width)
    ax.set_ylim(height, -1)
    ax.set_aspect('equal')
    fs.image_axes(ax)

def colour_lookup(values, cmap, norm):
    """color_of(structure_id, half) for a value per region (Series indexed by structure_id)."""
    def color_of(structure_id, half):
        value = values.get(structure_id, np.nan)
        return None if np.isnan(value) else cmap(norm(value))
    return color_of

def colorbar(fig, cmap, norm, axes, label, shrink=0.6):
    bar = fig.colorbar(ScalarMappable(norm, cmap), ax=np.ravel(axes).tolist(), shrink=shrink, pad=0.01)
    bar.set_label(label, color=fs.INK_2)
    bar.outline.set_visible(False)

def section_title(ax, index, loc='center'):
    ax.set_title(f'{position(index):.1f} mm', fontsize=10, fontweight='normal', color=fs.INK_2, loc=loc)

print(f'{plane} sections ({position_label}):', ', '.join(f'{position(i):.1f}' for i in sections))

### 4a. Average density of each group

One row per group, the same colour scale for both (0 to `vmax`; by default the 98th percentile of the group means,
so that a few very dense regions do not wash out the others; denser regions get the darkest colour).

In [ ]:
group_means = {group: density[mice[group]].mean(axis=1) for group in groups}
vmax = np.percentile(pd.concat(group_means.values()).dropna(), 98)
density_norm = Normalize(0, vmax, clip=True)

fig, axes = plt.subplots(len(groups), n_sections, squeeze=False,
                         figsize=(panel[0] * n_sections + 1.2, (panel[1] + 0.4) * len(groups) + 0.8))
for row, group in enumerate(groups):
    color_of = colour_lookup(group_means[group], fs.ORANGE_RAMP, density_norm)
    for col, index in enumerate(sections):
        draw_section(axes[row, col], shapes[index], color_of)
        if row == 0:
            section_title(axes[row, col], index)
    axes[row, 0].set_ylabel(f'{group}\n(n = {len(mice[group])})', color=fs.INK, fontsize=12, fontweight='bold')
fs.scale_bar(axes[-1, -1], 1000, atlas.resolution[0], label='1 mm', color=fs.INK_2)
colorbar(fig, fs.ORANGE_RAMP, density_norm, axes, density_unit)
fs.title(fig, f'Average {density_unit} per region, {plane} sections ({position_label})')
fs.save(fig, out, f'4a_atlas_density_per_group_{plane}')

### 4b. Both groups side by side

The same values as 4a (same colour scale), drawn so that the two groups touch:

- **sagittal**: every section twice, front to front: the first group on the left (mirrored, front on the right),
  the second group on the right.
- **horizontal and coronal**: one section shows both groups: the left half of each section is the first group, the
  right half the second (both halves show the average of the two hemispheres of the mice).

In [ ]:
lookups = {1: colour_lookup(group_means[first], fs.ORANGE_RAMP, density_norm),
           2: colour_lookup(group_means[second], fs.ORANGE_RAMP, density_norm)}

if plane == 'sagittal':
    pairs_per_row = 2
    n_rows = int(np.ceil(n_sections / pairs_per_row))
    size = 2.8                                         # width of one section (inches)
    gap = 0.25                                         # space between two pairs (in widths of a section)
    fig, axes = plt.subplots(n_rows, 3 * pairs_per_row - 1, squeeze=False,
                             width_ratios=([1, 1, gap] * pairs_per_row)[:-1],
                             figsize=((2 + gap) * pairs_per_row * size + 1.2,
                                      n_rows * (size * height / width + 0.4) + 0.8))
    for ax in axes.ravel():
        ax.set_visible(False)
    for number, index in enumerate(sections):
        row, pair = divmod(number, pairs_per_row)
        left, right = axes[row, 3 * pair], axes[row, 3 * pair + 1]
        left.set_visible(True)
        right.set_visible(True)
        draw_section(left, shapes[index], lookups[1])
        left.invert_xaxis()                            # mirrored: the front touches the other group's front
        draw_section(right, shapes[index], lookups[2])
        section_title(left, index, loc='right')
    for ax, group in [(axes[0, 0], first), (axes[0, 1], second)]:
        ax.text(0.5, -0.02, group, transform=ax.transAxes, ha='center', va='top', color=fs.INK, fontsize=10,
                fontweight='bold')
    fs.scale_bar(axes[0, 1], 1000, atlas.resolution[0], label='1 mm', color=fs.INK_2)
    colorbar(fig, fs.ORANGE_RAMP, density_norm, [ax for ax in axes.ravel() if ax.get_visible()], density_unit)
    fs.title(fig, f'Average {density_unit}: {first} (left, mirrored) and {second} (right), '
                  f'sagittal sections ({position_label})')
else:
    def split_colour(structure_id, half):
        return lookups[half](structure_id, half)

    fig, axes = plt.subplots(1, n_sections, figsize=(panel[0] * n_sections + 1.2, panel[1] + 1.0), squeeze=False)
    for ax, index in zip(axes[0], sections):
        draw_section(ax, split_shapes[index], split_colour)
        ax.axvline(middle - crop[1].start - 0.5, color=fs.INK_2, linewidth=0.6, linestyle='--')
        section_title(ax, index)
    for half, group in [(0.25, first), (0.75, second)]:
        axes[0, 0].text(half, -0.02, group, transform=axes[0, 0].transAxes, ha='center', va='top',
                        color=fs.INK, fontsize=10, fontweight='bold')
    fs.scale_bar(axes[0, -1], 1000, atlas.resolution[0], label='1 mm', color=fs.INK_2)
    colorbar(fig, fs.ORANGE_RAMP, density_norm, axes, density_unit, shrink=0.95)
    fs.title(fig, f'Average {density_unit}: {first} (left half) and {second} (right half), '
                  f'{plane} sections ({position_label})')
fs.save(fig, out, f'4b_atlas_density_side_by_side_{plane}')

### 4c. Difference between the groups

Every region coloured by the log2 ratio of the mean densities: orange = more cells in the second group, blue = more
in the first, gray = the same. The scale is symmetric (±`limit`, by default the 98th percentile of the absolute
log2 ratios). Regions with p < 0.05 have a dark outline.

In [ ]:
limit = np.nanpercentile(statistics.log2_ratio.abs(), 98)
ratio_norm = Normalize(-limit, limit, clip=True)
color_of = colour_lookup(statistics.log2_ratio, fs.BLUE_ORANGE, ratio_norm)
bold = set(statistics.index[statistics.p < 0.05])

fig, axes = plt.subplots(1, n_sections, figsize=(panel[0] * n_sections + 1.2, panel[1] + 1.0), squeeze=False)
for ax, index in zip(axes[0], sections):
    draw_section(ax, shapes[index], color_of, bold=bold)
    section_title(ax, index)
fs.scale_bar(axes[0, -1], 1000, atlas.resolution[0], label='1 mm', color=fs.INK_2)
colorbar(fig, fs.BLUE_ORANGE, ratio_norm, axes, f'log2 ratio\n{second} / {first}', shrink=0.95)
fs.title(fig, f'{second} vs {first}, {plane} sections ({position_label}): orange = more cells in {second}, '
              f'blue = more in {first}; dark outline = p < 0.05')
fs.save(fig, out, f'4c_atlas_difference_{plane}')

## 5. Reference: the Beryl regions

### 5a. Atlas sections with the name of every region

The sections of figure 4 (same `plane`), large (one figure per section), with every Beryl region outlined and its
acronym written inside it, to look up the regions of the maps. In horizontal and coronal sections the regions are
the same on both sides, so the acronyms are written on the right half of the image only (regions on the midline, or
only on the left, are named where they are). An acronym sits at the point deepest inside its region, and its size
follows the size of the region; zoom into the PDF for the small ones. Full names: figure 5b or `beryl_regions.csv`.

In [ ]:
from scipy import ndimage

def region_names(index):
    """(acronym, x, y, depth) of every Beryl region of a section: at the point deepest inside the region."""
    labels = fr.to_beryl(cut(annotation, index)[crop], atlas)
    right = np.arange(crop[1].start, crop[1].stop)[None, :] >= middle
    names = []
    for structure_id in np.unique(labels[labels > 0]):
        mask = labels == structure_id
        if plane != 'sagittal' and (mask & right).any():
            mask &= right
        depth = ndimage.distance_transform_edt(np.pad(mask, 1))[1:-1, 1:-1]
        y, x = np.unravel_index(np.argmax(depth), depth.shape)
        names.append((beryl.acronym[structure_id], x, y, depth[y, x]))
    return names

for number, index in enumerate(sections):
    fig, ax = plt.subplots(figsize=(11, 11 * height / width + 0.5))
    draw_section(ax, shapes[index], lambda structure_id, half: None)
    for acronym, x, y, depth in region_names(index):
        ax.text(x, y, acronym, ha='center', va='center', fontsize=np.clip(3 + 0.5 * depth, 4, 9), color=fs.INK)
    fs.title(fig, f'Beryl regions, {plane} section {position(index):.1f} {position_label}')
    fs.save(fig, out, f'5a_regions_{plane}_{number}_{position(index):.1f}mm')

### 5b. List of all Beryl regions

All 306 Beryl regions in Allen order, by major division: acronym and full name. Regions left out of the analysis
(smaller than `min_volume_mm3`) are in gray. The same list is saved as `beryl_regions.csv` (with the structure id and
a column `analysed`).

In [ ]:
all_beryl = beryl.assign(analysed=beryl.index.isin(density.index))
all_beryl.to_csv(out / 'beryl_regions.csv')

lines = []   # (acronym, name, kind): kind = 'division', 'region', 'left out' or 'gap'
for division in fr.COSMOS:
    table = all_beryl[all_beryl.division == division]
    title = fr.COSMOS_NAMES[division] + (f' ({division})' if fr.COSMOS_NAMES[division] != division else '')
    lines.append(('', f'{title}: {len(table)} regions', 'division'))
    lines += [(r.acronym, r.name, 'region' if r.analysed else 'left out') for r in table.itertuples()]
    lines.append(('', '', 'gap'))

# fill columns top to bottom; a division title never ends a column, a gap never starts one
n_columns = 4
per_column = int(np.ceil(len(lines) / n_columns)) + 2
columns = [[]]
for line in lines:
    if len(columns[-1]) >= per_column or (line[2] == 'division' and len(columns[-1]) >= per_column - 1):
        columns.append([])
    if line[2] == 'gap' and not columns[-1]:
        continue
    columns[-1].append(line)

fig, ax = plt.subplots(figsize=(4.3 * len(columns), 0.135 * per_column + 0.8))
for column, column_lines in enumerate(columns):
    for row, (acronym, name, kind) in enumerate(column_lines):
        if kind == 'division':
            ax.text(column, row, name, fontsize=8, fontweight='bold', color=fs.INK, va='center')
        elif kind != 'gap':
            color = fs.INK_2 if kind == 'region' else fs.MUTED
            ax.text(column, row, acronym, fontsize=7, fontweight='bold', color=color, va='center')
            ax.text(column + 0.17, row, name + ('  (left out: too small)' if kind == 'left out' else ''),
                    fontsize=7, color=color, va='center')
ax.set_xlim(-0.02, len(columns))
ax.set_ylim(per_column, -1)
ax.axis('off')
fs.title(fig, f'The {len(all_beryl)} Beryl regions (International Brain Laboratory), by major division')
fs.save(fig, out, '5b_beryl_regions_list')